# Landsat 8/9 Processing Workflow

This notebook demonstrates the complete workflow for processing Landsat 8/9 satellite imagery using the `disasters-product-algorithms` package.

## Workflow Steps
1. **Configure Environment Variables** - Set processing parameters
2. **Process Landsat Data** - Generate products with COG conversion and event naming
3. **View Results** - Examine the generated outputs
4. **Upload to S3** - Upload processed files to the disasters bucket (optional)

## Features Demonstrated
- Reads Landsat surface-reflectance imagery directly from a STAC API (Earth Search, collection) and its cloud-optimized assets on S3 (s3://usgs-landsat).
- Cloud Optimized GeoTIFF (COG) conversion with customizable compression
- Event-based file naming for disaster response
- Multiple product generation (true color, natural color, NDVI, NDWI, MNDWI, EVI, NBR)
- Cloud masking
- Automatic merging by date and product

## 1. Environment Setup

Configure all processing parameters as environment variables for easy modification.

In [ ]:
import sys
import os

# Tell Python where to find shared utilities and the Landsat package
sys.path.append("../src")

# ==============================================================================
# ACTIVATION OPTIONS BLOCK
# Change these variables for each new disaster activation.
# ==============================================================================

# Output
OUTPUT_DIR = "/tmp/s3_temp"

# Event metadata
EVENT_NAME = "202610_Hurricane_Isaias"
SOURCE = "USGS"
PROCESS_DATE = None
PROCESS_TILE = None

# Landsat STAC search parameters
START_DATE = "2026-10-09"
END_DATE = "2026-10-12"
BBOX = [-88.00, 35.00, -86.00, 36.00]
CLOUD_COVER = 100

# Products supported by process_landsat_odr.py
# Products supported by process_landsat_odr.py
PRODUCTS = {
    "true_color": True,
    "natural_color": False,
    "color_infrared": False,
    "swir": False,
    "ndvi": False,
    "ndwi": False,
    "mndwi": False,
    "evi": False,
    "nbr": False,
}

# COG options
COMPRESSION = "ZSTD"
COMPRESSION_LEVEL = 9
NODATA = None
TIF_ONLY = False

# Processing flags
ENABLE_MERGE = True
ENABLE_MASK = True
FORCE_OVERWRITE = False
UNZIP_ONLY = False
TARGET_CRS = None

# S3 upload settings
ENABLE_S3_UPLOAD = False

# Destination configuration
from shared_utils.product_paths import STAGING_BUCKET

S3_BUCKET = STAGING_BUCKET
PRODUCT_SENSOR = "landsat"

In [ ]:
import os
import json
import tempfile
from shared_utils import PROCESSOR_STRING

# Ensure the Landsat working directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ACTIVATION METADATA

ACTIVATION_METADATA = {
    "ACTIVATION_EVENT": EVENT_NAME,
    "SOURCE": SOURCE,
    "PROCESSOR": PROCESSOR_STRING,
}

# Write activation metadata to a temporary JSON file
# Reuse one file for Landsat rather than creating a new file each run
ACTIVATION_METADATA_PATH = os.path.join(
    tempfile.gettempdir(),
    "activation_meta_landsat.json"
)

with open(ACTIVATION_METADATA_PATH, "w") as f:
    json.dump(ACTIVATION_METADATA, f, indent=2)

print(f"Configuration loaded for event: {EVENT_NAME}")
print(f"Landsat working directory: {OUTPUT_DIR}")
print(f"Activation metadata: {ACTIVATION_METADATA_PATH}")

## 2. Verify Input Data

Check that Landsat files are present in the input directory.

In [ ]:
import os

# CHECK LANDSAT PROCESSING CONFIGURATION
print(f"Landsat working directory: {OUTPUT_DIR}")
print(f"Event: {EVENT_NAME}")
print(f"Date range: {START_DATE} to {END_DATE}")

if BBOX is not None:
    print(f"Bounding box (min_lon, min_lat, max_lon, max_lat): {BBOX}")
else:
    print("WARNING: BBOX is not set. Define BBOX before running the Landsat STAC search.")

print("\nProducts selected:")
for product in PRODUCTS:
    print(f"  - {product}")

print("\n✓ Configuration checked.")

## 3. Preview the area of interest

Optional. Plots the bounding box against the Landsat tiles it
intersects, so you can see how much imagery a run will pull.

In [ ]:
# ==============================================================================
# PREVIEW THE AOI
# ==============================================================================
# Each matched Landsat scene can cover a large area (~185 km across).
# The red box shows the AOI; blue boxes show matching Landsat scene footprints.
# This is a preview only and does not run the processing workflow.

try:
    import matplotlib.pyplot as plt
    from pystac_client import Client

    catalog = Client.open("https://earth-search.aws.element84.com/v1")
    collection = "landsat-c2-l2"

    search = catalog.search(
        collections=[collection],
        datetime=f"{START_DATE}/{END_DATE}",
        bbox=BBOX,
        query={"eo:cloud_cover": {"lt": CLOUD_COVER}},
        max_items=50,
    )

    items = list(search.items())

    print(f"{len(items)} Landsat scene(s) match:")

    for item in items:
        cloud = item.properties.get("eo:cloud_cover")
        cloud_text = f"{cloud:.1f}%" if cloud is not None else "unknown"
        print(f"  {item.id}  cloud={cloud_text}")

    fig, ax = plt.subplots(figsize=(8, 8))

    for item in items:
        b = item.bbox

        ax.add_patch(
            plt.Rectangle(
                (b[0], b[1]),
                b[2] - b[0],
                b[3] - b[1],
                fill=False,
                edgecolor="tab:blue",
                linewidth=1,
            )
        )

        # Landsat IDs look like LC09_L2SP_018040_20240923_02_T1.
        # Show the satellite and WRS-2 path/row identifier.
        parts = item.id.split("_")
        label = "_".join(parts[:3]) if len(parts) >= 3 else item.id

        ax.text(
            b[0], b[3], label,
            fontsize=6,
            color="tab:blue",
        )

    # Draw the AOI on top of the scene footprints.
    ax.add_patch(
        plt.Rectangle(
            (BBOX[0], BBOX[1]),
            BBOX[2] - BBOX[0],
            BBOX[3] - BBOX[1],
            fill=False,
            edgecolor="red",
            linewidth=2,
            label="AOI",
        )
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("AOI and matching Landsat scene footprints")
    ax.legend()
    ax.autoscale_view()
    ax.set_aspect("auto")
    plt.tight_layout()
    plt.show()

except ImportError as exc:
    print(f"Preview skipped ({exc}). Processing does not need this cell.")


## 4. Process Landsat Data

Process the Landsat imagery to generate various products with COG conversion and event naming.

**Note:** The processing script has been configured with unbuffered output to display progress in real-time within JupyterHub. You'll see:
- Progress bars for scene processing
- Detailed product generation steps
- COG conversion progress
- Error messages if any products fail
- Final processing summary with success/failure counts
- Log file location for detailed error tracking

In [ ]:
import os
import sys
import subprocess

PROCESS_SCRIPT = os.path.join(
    "..", "src", "landsat", "process_landsat_odr.py"
)

if not os.path.isfile(PROCESS_SCRIPT):
    raise FileNotFoundError(
        f"Could not find Landsat processing script: {PROCESS_SCRIPT}"
    )

if not START_DATE or not END_DATE:
    raise ValueError("START_DATE and END_DATE must be specified.")

if BBOX is None or len(BBOX) != 4:
    raise ValueError(
        "BBOX must be [min_lon, min_lat, max_lon, max_lat]."
    )

# Map legacy product names to the names supported by the new CLI.
PRODUCT_MAP = {
    "true": "true_color",
    "nat": "natural_color",
    "ndvi": "ndvi",
    "ndwi": "ndwi",
    "mndwi": "mndwi",
    "evi": "evi",
    "nbr": "nbr",
    "true_color": "true_color",
    "natural_color": "natural_color",
    "color_infrared": "color_infrared",
    "swir": "swir",
}

# Select only products explicitly enabled in the dictionary.
enabled_products = [
    product
    for product, enabled in PRODUCTS.items()
    if enabled
]

if not enabled_products:
    raise ValueError(
        "No Landsat products are enabled. Set at least one "
        "entry in PRODUCTS to True."
    )

# Translate enabled products to the names supported by the CLI.
mapped_products = []

for product in enabled_products:
    if product not in PRODUCT_MAP:
        raise ValueError(
            f"Unrecognized Landsat product: {product}"
        )

    mapped = PRODUCT_MAP[product]

    if mapped not in mapped_products:
        mapped_products.append(mapped)

# Ensure output directory exists.
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Starting Landsat STAC processing...")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Bounding box: {BBOX}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Enabled products: {mapped_products}")

# Process each product separately because --product accepts one value.
overall_success = True

for product in mapped_products:
    process_cmd = [
        sys.executable,
        PROCESS_SCRIPT,
        "--product", product,
        "--start-date", START_DATE,
        "--end-date", END_DATE,
        "--bbox", *[str(value) for value in BBOX],
        "--cloud-cover", str(CLOUD_COVER),
        "--output", OUTPUT_DIR,
        "--metadata-json", ACTIVATION_METADATA_PATH,
    ]

    if ENABLE_MASK:
        process_cmd.append("--cloud-mask")

    if ENABLE_MERGE:
        process_cmd.append("--merge")

    print("\n" + "=" * 70)
    print(f"Processing product: {product}")
    print(f"Command: {' '.join(process_cmd)}")
    print("=" * 70)

    process = subprocess.Popen(
        process_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")

    return_code = process.wait()

    if return_code == 0:
        print(f"\n✓ {product} completed successfully.")
    else:
        print(
            f"\n✗ {product} failed with return code {return_code}."
        )
        overall_success = False

if overall_success:
    print("\n✓ All enabled Landsat products processed successfully!")
else:
    raise RuntimeError(
        "One or more Landsat products failed. "
        "Review the output above."
    )


## 5. View Results

Examine the generated output files and directory structure.

In [ ]:
import os
import glob

# ==============================================================================
# FIND AND DISPLAY GENERATED LANDSAT PRODUCTS
# ==============================================================================

# The new processing command writes products directly to INPUT_DIR.
output_dir = OUTPUT_DIR

if os.path.exists(output_dir):
    print(f"Output directory: {output_dir}\n")
    print("Generated Products:")
    print("=" * 80)

    # Search recursively in case the directory contains nested outputs.
    tif_files = sorted(
        glob.glob(
            os.path.join(output_dir, "**", "*.tif"),
            recursive=True,
        )
    )

    # Exclude cloud-mask sidecars or other internal intermediates if present.
    tif_files = [
        path for path in tif_files
        if os.path.isfile(path)
        and "_cloud_masks" not in path.split(os.sep)
    ]

    if tif_files:
        # Group products using the product name in each filename.
        product_types = {}

        for tif_file in tif_files:
            filename = os.path.basename(tif_file)
            parts = filename.split("_")

            # Landsat names typically contain SAT_LEVEL_product_...
            product_type = parts[2] if len(parts) >= 3 else "unknown"
            product_types.setdefault(product_type, []).append(tif_file)

        for product_type, paths in sorted(product_types.items()):
            print(f"\n{product_type}:")

            for path in sorted(paths):
                file_size = os.path.getsize(path) / (1024 * 1024)
                print(f"  - {os.path.basename(path)} ({file_size:.1f} MB)")

        print(f"\nTotal GeoTIFF files: {len(tif_files)}")
        print(
            f"Total size: "
            f"{sum(os.path.getsize(p) for p in tif_files) / (1024**3):.2f} GB"
        )
    else:
        print("No GeoTIFF files found.")
else:
    print(f"Output directory not found: {output_dir}")

## Output File Naming Convention

**Original:** `LC08_trueColor_20250922_185617_046028.tif`

**Final:** `LC08_trueColor_046028_2025-09-22T18:56:17Z.tif`

**If Merged:** `LC08_trueColor_merged_2025-09-22T18:56:17Z.tif`

**If Masked:** `LC08_trueColor_046028_masked_2025-09-22T18:56:17Z.tif`

**If Merged and Masked:** `LC08_trueColor_merged_masked_2025-09-22T18:56:17Z.tif`

This naming convention:
- Moves the tile number (046028)
- Adds formatted datetime (YYYY-MM-DDTHH:MM:SSZ) at the end
- Includes `_merged_` and/or `_masked_` when applicable

## Product Descriptions

| Product | Description | Use Case |
|---------|-------------|----------|
| **true** | True color (RGB) | Visual interpretation |
| **nat** | Natural color | Natural-looking imagery |
| **colorIR** | Color infrared | Vegetation health |
| **ndvi** | Normalized Difference Vegetation Index | Vegetation density |
| **ndwi** | Normalized Difference Water Index | Water detection |
| **mndwi** | Modified NDWI | Enhanced water detection |
| **evi** | Enhanced Vegetation Index | Improved vegetation monitoring |
| **nbr** | Normalized Burn Ratio | Burn severity assessment |

## Next Steps

You can now:
1. Load and visualize the GeoTIFF files using libraries like `rasterio` or `GDAL`
2. Upload the COG files to cloud storage (S3, GCS, etc.)
3. Process additional dates or tiles by modifying the configuration variables
4. Generate additional products by updating the `PRODUCTS` list
5. Experiment with different compression settings for optimal file sizes

## 6. Plot Results

Render a quick preview of each generated product (one sample per product type) to sanity-check the output before uploading. Plots are in grayscale and RGB images render only one band. 

In [ ]:
# ==============================================================================
# PREVIEW THE LANDSAT COGs
# ==============================================================================
from pathlib import Path
from shared_utils.plotting import preview_cogs

# Landsat process_landsat_odr.py writes products directly to INPUT_DIR
LANDSAT_OUTPUT_DIR = Path(OUTPUT_DIR)

cogs = sorted([
    str(path)
    for path in LANDSAT_OUTPUT_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".tif", ".tiff"}
    and "_cloud_masks" not in path.parts
])

if cogs:
    print(f"Previewing {len(cogs)} Landsat COG(s)...")
    preview_cogs(cogs, sample_n=10)
else:
    print(f"No Landsat COGs found in {LANDSAT_OUTPUT_DIR}.")

# Upload to S3 (Optional) #

Publish the finished COGs. **Opt-in** — runs only when `ENABLE_S3_UPLOAD = True` in the configuration cell.

Uploads go to `s3://{S3_BUCKET}/ProgramData/Landsat/<Product>/<filename>` — the same bucket, prefix and key layout a **DPS job** produces, and keyed by the same code (`shared_utils.staging_upload`). The `<date>/` level groups scenes for merging on disk and is dropped from the key; each product is published to its own hard-coded destination from `shared_utils.product_paths`.

The upload uses **ambient AWS credentials** — on the hub the pod already assumes `disasters-prod`. A real write probe runs against every destination prefix before anything is uploaded, so a missing grant fails immediately instead of part-way through. When merging, only the merged COGs are published.

In [ ]:
# ==============================================================================
# UPLOAD TO S3 (optional)
# ==============================================================================
# Publishes Landsat COGs individually using ambient AWS credentials.

import os
from pathlib import Path

from shared_utils import upload_file_to_s3
from shared_utils.product_paths import prefix_for_product_dir, product_dir

# Filename product token -> canonical product key in shared_utils.product_paths.
# These tokens match the Landsat output filenames.
_PRODUCT_KEYS = {
    "trueColor": "trueColor",
    "naturalColor": "naturalColor",
    "colorInfrared": "colorInfrared",
    "swir": "shortwaveInfrared",
    "ndvi": "NDVI",
    "ndwi": "NDWI",
    "mndwi": "MNDWI",
    "nbr": "NBR",
    "evi": "EVI",
}

PRODUCT_FOLDERS = {
    token: product_dir(PRODUCT_SENSOR, key)
    for token, key in _PRODUCT_KEYS.items()
}

_enabled_products = set(PRODUCTS)

_unmapped = _enabled_products - {
    "true_color",
    "natural_color",
    "color_infrared",
    "swir",
    "ndvi",
    "ndwi",
    "mndwi",
    "nbr",
    "evi",
}

if _unmapped:
    raise ValueError(
        f"Products with no folder mapping: {sorted(_unmapped)}"
    )

if _unmapped:
    raise ValueError(
        f"Products with no folder mapping: {sorted(_unmapped)}"
    )

if not ENABLE_S3_UPLOAD:
    print(
        "S3 upload OFF. Review the outputs above, then set "
        "ENABLE_S3_UPLOAD = True in the config cell to publish."
    )

else:
    # Use NEW_COGS if your processing cell records files generated in
    # this run. Otherwise, this fallback scans the output directory.
    if "NEW_COGS" in globals():
        cogs = [str(f) for f in NEW_COGS]
    else:
        cogs = [
            str(f)
            for f in Path(OUTPUT_DIR).glob("*.tif")
            if f.is_file()
        ]

    # Only publish actual product COGs, not temporary files or cloud masks.
    cogs = [
        f for f in cogs
        if not f.endswith(".tmp.tif")
        and "_cloud_masks" not in Path(f).parts
    ]

    if ENABLE_MERGE:
        cogs = [
            f for f in cogs
            if "merged" in os.path.basename(f)
        ]

    uploaded = 0
    skipped = []

    for f in cogs:
        filename = os.path.basename(f)

        # Find the product token in the Landsat filename.
        # Sort longest-first to avoid partial-token matches.
        subfolder = None
        for token in sorted(PRODUCT_FOLDERS, key=len, reverse=True):
            if f"_{token}_" in filename:
                subfolder = PRODUCT_FOLDERS[token]
                break

        if subfolder is None:
            skipped.append(filename)
            continue

        s3_uri = (
            f"s3://{S3_BUCKET}/"
            f"{prefix_for_product_dir(PRODUCT_SENSOR, subfolder)}/"
            f"{filename}"
        )

        upload_file_to_s3(f, s3_uri)
        print(f"Uploaded: {filename}")
        uploaded += 1

    print(f"\nUploaded {uploaded} Landsat COG(s).")
    print(f"Destination: s3://{S3_BUCKET}/ProgramData/<Sensor>/<Product>/")

    if skipped:
        raise RuntimeError(
            f"{len(skipped)} COG(s) had no product-folder mapping "
            f"and were NOT uploaded: {skipped}"
        )